# fabric_icm_dbt - orchestration run

Wordt aangeroepen door een **Notebook-activiteit** in een Fabric Data Pipeline (zie `orchestration/README.md`).

Stappen:
1. Clone/pull de dbt-repo (git).
2. Installeer dependencies.
3. Haal Fabric service principal secrets op uit een Key Vault.
4. Draai `dbt seed` + `dbt build`.
5. Laat de notebook falen als dbt faalt, zodat de pipeline-activiteit rood wordt en de pipeline-run/notificatie dat oppikt.

In [ ]:
# Parameters (kunnen als pipeline-parameters overschreven worden via een 'parameters' cell)
git_repo_url = "https://github.com/AbdullahOzisik/fabric-icm-dbt.git"
git_branch = "main"
key_vault_url = "https://<jouw-keyvault-naam>.vault.azure.net/"
target_schema = "prod"

In [ ]:
import os
import shutil
import subprocess

repo_dir = "/tmp/fabric_icm_dbt"
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

subprocess.run(["git", "clone", "--branch", git_branch, git_repo_url, repo_dir], check=True)

In [ ]:
subprocess.run(["pip", "install", "-r", f"{repo_dir}/requirements.txt"], check=True)

In [ ]:
# notebookutils is beschikbaar in Fabric-notebooks (geen losse install nodig).
# Haalt de service-principal secrets op uit een Key Vault die aan deze workspace gekoppeld is.
from notebookutils import mssparkutils

os.environ["FABRIC_SERVER"] = mssparkutils.credentials.getSecret(key_vault_url, "fabric-server")
os.environ["FABRIC_DATABASE"] = mssparkutils.credentials.getSecret(key_vault_url, "fabric-database")
os.environ["FABRIC_SCHEMA"] = target_schema
os.environ["FABRIC_CLIENT_ID"] = mssparkutils.credentials.getSecret(key_vault_url, "fabric-client-id")
os.environ["FABRIC_CLIENT_SECRET"] = mssparkutils.credentials.getSecret(key_vault_url, "fabric-client-secret")
os.environ["FABRIC_TENANT_ID"] = mssparkutils.credentials.getSecret(key_vault_url, "fabric-tenant-id")
os.environ["DBT_PROFILES_DIR"] = f"{repo_dir}/profiles"

In [ ]:
result_deps = subprocess.run(["dbt", "deps"], cwd=repo_dir)
result_seed = subprocess.run(["dbt", "seed"], cwd=repo_dir)
result_build = subprocess.run(["dbt", "build", "--target", "ci"], cwd=repo_dir)

if result_deps.returncode != 0 or result_seed.returncode != 0 or result_build.returncode != 0:
    raise RuntimeError("dbt run gefaald - zie de logs hierboven. Notebook faalt bewust zodat de pipeline dit als fout ziet.")